In [1]:
import nilearn as nl
import numpy as np
import matplotlib
import pandas as pd
import os
import nibabel as nib
import matplotlib.pyplot as plt
import openpyxl as xl

from nilearn.glm.first_level import make_first_level_design_matrix
from nilearn.plotting import plot_design_matrix, plot_contrast_matrix
from nilearn.glm.first_level import FirstLevelModel
from nilearn.masking import compute_epi_mask, apply_mask, unmask
from nilearn.image import index_img, resample_to_img, math_img, mean_img, get_data
from nilearn.plotting import plot_stat_map, show
from nilearn import image, masking
from nilearn import plotting
from nilearn.datasets import load_mni152_brain_mask
from pandas import read_excel

from nilearn.glm.second_level import SecondLevelModel
from scipy.stats import norm

from nilearn.glm import threshold_stats_img
from pathlib import Path
from atlasreader import create_output

The Python package you are importing, AtlasReader, is licensed under the
BSD-3 license; however, the atlases it uses are separately licensed under more
restrictive frameworks.
By using AtlasReader, you agree to abide by the license terms of the
individual atlases. Information on these terms can be found online at:
https://github.com/miykael/atlasreader/tree/master/atlasreader/data



# fcontrast - test moral vs non-moral for each story separately

In [2]:
stories = ["tunnel","21styear","prettymouth","shapessocial","shapesphysical"]
virtueORvice = 'virtues'

# output directory
base_dir1 = os.path.expanduser(
    "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/"
)

output_dir = Path(f"{base_dir1}{virtueORvice}/minStatConj/")


In [3]:

for story in stories:
    print(f"\nProcessing story: {story}")
    
    if virtueORvice == 'virtues':
        virtueORviceDataFolder = '7_MAC'
    
    if virtueORvice == 'vices':
        virtueORviceDataFolder = '7_MAC_V'
    
    base_dir2 = os.path.expanduser(
        "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/processed_first_level_per_sentence/"
    )
    
    fcontrast_dir = Path(f"{base_dir2}{story}/{virtueORviceDataFolder}/Conjunction/")
    
    subfolder = os.path.join(fcontrast_dir)
    files = [f for f in os.listdir(subfolder) if f.endswith(".nii") or f.endswith(".nii.gz")]
    full_paths = [os.path.join(subfolder, f) for f in files]
    
    # Sanity check
    if not full_paths:
        print(f"No .nii or .nii.gz files found in {subfolder}. Skipping...")
        continue
    
    # Create design matrix
    design_matrix = pd.DataFrame(
        [1] * len(full_paths),
        columns=["intercept"],
    )

    # Fit the second-level model
    second_level_model = SecondLevelModel()
    second_level_model = second_level_model.fit(
        full_paths,
        design_matrix=design_matrix,
    )

    # Compute z-map
    z_map = second_level_model.compute_contrast(
        second_level_contrast="intercept",
        output_type="z_score",
    )

    # Save z-map in subfolder
    z_map_filename = os.path.join(output_dir, f"{story}_zscores_minStatConj_secondlevel.nii.gz")
    z_map.to_filename(z_map_filename)
    print(f"✅ Saved z-map to: {z_map_filename}")

    # Threshold map
    thresholded_map, threshold = threshold_stats_img(
        stat_img=z_map,
        alpha=0.05,
        height_control='bonferroni', #bonferroni,fdr
        cluster_threshold=0, #2,5,10,0
        two_sided=True,
    )

    # Save thresholded map in subfolder
    thresh_filename = os.path.join(output_dir,f"{story}_threshold_{threshold:.3g}_bonfcorrect_zscores_minStatConj_secondlevel.nii.gz")
    thresholded_map.to_filename(thresh_filename)
    print(f"✅ Saved thresholded map to: {thresh_filename}")

    # Threshold map
    thresholded_map, threshold = threshold_stats_img(
        stat_img=z_map,
        alpha=0.05,
        height_control='fdr', #bonferroni,fdr
        cluster_threshold=0, #2,5,10,0
        two_sided=True,
    )

    # Save thresholded map in subfolder
    thresh_filename = os.path.join(output_dir,f"{story}_threshold_{threshold:.3g}_fdrcorrect_zscores_minStatConj_secondlevel.nii.gz")
    thresholded_map.to_filename(thresh_filename)
    print(f"✅ Saved thresholded map to: {thresh_filename}")

    output_path2 = os.path.join(output_dir,f"{story}_threshold_{threshold:.3g}_fdrcorrect_zscores_minStatConj_secondlevel.nii.gz")

    create_output(
            output_path2,
            cluster_extent=0,
            direction="pos",
            outdir=str(output_dir)
        )

    # Threshold map
    thresholded_map, threshold = threshold_stats_img(
        stat_img=z_map,
        alpha=0.05,
        height_control='fpr', #bonferroni,fdr
        cluster_threshold=5, #2,5,10,0
        two_sided=True,
    )

    # Save thresholded map in subfolder
    thresh_filename = os.path.join(output_dir,f"{story}_threshold_{threshold:.3g}_fprcorrectClustThr5_zscores_minStatConj_secondlevel.nii.gz")
    thresholded_map.to_filename(thresh_filename)
    print(f"✅ Saved thresholded map to: {thresh_filename}")

    # Threshold map
    thresholded_map, threshold = threshold_stats_img(
        stat_img=z_map,
        alpha=0.05,
        height_control='fpr', #bonferroni,fdr
        cluster_threshold=10, #2,5,10,0
        two_sided=True,
    )

    # Save thresholded map in subfolder
    thresh_filename = os.path.join(output_dir,f"{story}_threshold_{threshold:.3g}_fprcorrectClustThr10_zscores_minStatConj_secondlevel.nii.gz")
    thresholded_map.to_filename(thresh_filename)
    print(f"✅ Saved thresholded map to: {thresh_filename}")

    output_path3 = os.path.join(output_dir,f"{story}_threshold_{threshold:.3g}_fprcorrectClustThr10_zscores_minStatConj_secondlevel.nii.gz")

    create_output(
            output_path3,
            cluster_extent=0,
            direction="pos",
            outdir=str(output_dir)
        )




Processing story: tunnel
✅ Saved z-map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/tunnel_zscores_minStatConj_secondlevel.nii.gz
✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/tunnel_threshold_4.91_bonfcorrect_zscores_minStatConj_secondlevel.nii.gz
✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/tunnel_threshold_2.02_fdrcorrect_zscores_minStatConj_secondlevel.nii.gz


/opt/anaconda3/envs/renv/lib/python3.12/site-packages/atlasreader/atlasreader.py:496: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  thresh_img = image.threshold_img(stat_img, threshold=voxel_thresh)
/opt/anaconda3/envs/renv/lib/python3.12/site-packages/nilearn/plotting/displays/_slicers.py:313: UserWarning: empty mask
  ims = self._map_show(img, type="imshow", threshold=threshold, **kwargs)


✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/tunnel_threshold_1.96_fprcorrectClustThr5_zscores_minStatConj_secondlevel.nii.gz
✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/tunnel_threshold_1.96_fprcorrectClustThr10_zscores_minStatConj_secondlevel.nii.gz


/opt/anaconda3/envs/renv/lib/python3.12/site-packages/atlasreader/atlasreader.py:496: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  thresh_img = image.threshold_img(stat_img, threshold=voxel_thresh)
/opt/anaconda3/envs/renv/lib/python3.12/site-packages/nilearn/plotting/displays/_slicers.py:313: UserWarning: empty mask
  ims = self._map_show(img, type="imshow", threshold=threshold, **kwargs)



Processing story: 21styear
✅ Saved z-map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/21styear_zscores_minStatConj_secondlevel.nii.gz
✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/21styear_threshold_4.91_bonfcorrect_zscores_minStatConj_secondlevel.nii.gz
✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/21styear_threshold_2.01_fdrcorrect_zscores_minStatConj_secondlevel.nii.gz


...

/opt/anaconda3/envs/renv/lib/python3.12/site-packages/atlasreader/atlasreader.py:496: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  thresh_img = image.threshold_img(stat_img, threshold=voxel_thresh)
/opt/anaconda3/envs/renv/lib/python3.12/site-packages/nilearn/plotting/displays/_slicers.py:313: UserWarning: empty mask
  ims = self._map_show(img, type="imshow", threshold=threshold, **kwargs)



Processing story: shapesphysical
✅ Saved z-map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/shapesphysical_zscores_minStatConj_secondlevel.nii.gz
✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/shapesphysical_threshold_5.23_bonfcorrect_zscores_minStatConj_secondlevel.nii.gz
✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/shapesphysical_threshold_2.01_fdrcorrect_zscores_minStatConj_secondlevel.nii

/opt/anaconda3/envs/renv/lib/python3.12/site-packages/atlasreader/atlasreader.py:496: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  thresh_img = image.threshold_img(stat_img, threshold=voxel_thresh)
/opt/anaconda3/envs/renv/lib/python3.12/site-packages/nilearn/plotting/displays/_slicers.py:313: UserWarning: empty mask
  ims = self._map_show(img, type="imshow", threshold=threshold, **kwargs)


✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/shapesphysical_threshold_1.96_fprcorrectClustThr5_zscores_minStatConj_secondlevel.nii.gz
✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/minStatConj/shapesphysical_threshold_1.96_fprcorrectClustThr10_zscores_minStatConj_secondlevel.nii.gz


/opt/anaconda3/envs/renv/lib/python3.12/site-packages/atlasreader/atlasreader.py:496: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  thresh_img = image.threshold_img(stat_img, threshold=voxel_thresh)
/opt/anaconda3/envs/renv/lib/python3.12/site-packages/nilearn/plotting/displays/_slicers.py:313: UserWarning: empty mask
  ims = self._map_show(img, type="imshow", threshold=threshold, **kwargs)
